### Модуль извлечения полного набора признаков для задачи распознавания степени усталости

In [2]:
import librosa
import numpy as np
from scipy.signal import find_peaks
from tqdm import tqdm
import os
from pathlib import Path
import math


SR = 16000                # частота дискретизации
HOP_LEN = 160             # шаг 10 мс
WIN_LEN = 400             # окно 25 мс
N_MFCC = 13               # количество MFCC коэффициентов
FRAME_LEN = 101           # число временных отсчётов (1 секунда)


def extract_mfcc_with_deltas(audio, sr=SR):
    """
    Извлекает MFCC-признаки, а также их дельты и дельта-дельты.

    Аргументы:
        audio (np.ndarray): Аудиосигнал (1D массив).
        sr (int): Частота дискретизации.

    Возвращает:
        np.ndarray: Массив формата (число фреймов, 39).
    """
    mfcc = librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=N_MFCC,
        hop_length=HOP_LEN, n_fft=WIN_LEN
    )
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    features = np.vstack([mfcc, delta, delta2])
    return features.T  # (101, 39)

def extract_f0(audio, sr=SR):
    """
    Извлекает траекторию основной частоты (f0).

    Аргументы:
        audio (np.ndarray): Аудиосигнал.
        sr (int): Частота дискретизации.

    Возвращает:
        np.ndarray: f0 в формате (число фреймов, 1).
    """
    f0, voiced_flag, voiced_probs = librosa.pyin(
        audio, fmin=80, fmax=300, sr=sr,
        hop_length=HOP_LEN, frame_length=450
    )
    f0 = np.nan_to_num(f0, nan=0.0)
    return f0.reshape(-1, 1)

def extract_rms(audio, sr=SR):
    """
    Выделяет среднеквадратичную энергию сигнала (RMS).

    Аргументы:
        audio (np.ndarray): Аудиосигнал.
        sr (int): Частота дискретизации.

    Возвращает:
        np.ndarray: RMS в формате (число фреймов, 1).
    """
    rms = librosa.feature.rms(y=audio, hop_length=HOP_LEN, frame_length=WIN_LEN)[0]
    return rms.reshape(-1, 1)

def extract_hnr(audio, sr=SR):
    """
    Вычисляет отношение энергий гармоники к остаточному шуму (HNR).

    Аргументы:
        audio (np.ndarray): Аудиосигнал.
        sr (int): Частота дискретизации.

    Возвращает:
        np.ndarray: Массив значений HNR (FRAME_LEN, 1).
    """
    harmonic = librosa.effects.harmonic(y=audio, margin=4.0)
    noise = audio - harmonic
    e_h = float(np.mean(harmonic ** 2) + 1e-9)  # энергия гармоники
    e_n = float(np.mean(noise ** 2) + 1e-9)     # остаточный шум
    return np.full((FRAME_LEN, 1), 10.0 * math.log10(e_h / e_n))

def extract_zcr(audio, sr=SR):
    """
    Вычисляет коэффициент пересечения нуля (ZCR).

    Аргументы:
        audio (np.ndarray): Аудиосигнал.
        sr (int): Частота дискретизации.

    Возвращает:
        np.ndarray: ZCR в формате (число фреймов, 1).
    """
    zcr = librosa.feature.zero_crossing_rate(y=audio, hop_length=HOP_LEN, frame_length=WIN_LEN)[0]
    return zcr.reshape(-1, 1)

def extract_jitter_shimmer(audio, sr=SR):
    """
    Вычисляет джиттер и шиммер аудиозаписи.

    Аргументы:
        audio (np.ndarray): Аудиосигнал.
        sr (int): Частота дискретизации.

    Возвращает:
        tuple[float, float]: (jitter, shimmer) - относительное изменение периода и амплитуды.
    """
    f0, voiced_flag, _ = librosa.pyin(
        audio, fmin=80, fmax=300, sr=sr,
        hop_length=HOP_LEN, frame_length=450
    )
    if voiced_flag is None:
        return 0.0, 0.0
    voiced_flag = np.asarray(voiced_flag, dtype=bool)
    if np.count_nonzero(voiced_flag) == 0:
        return 0.0, 0.0

    periods = 1.0 / f0[voiced_flag]
    if len(periods) < 3:
        return 0.0, 0.0

    period_diffs = np.abs(np.diff(periods))
    jitter = np.mean(period_diffs) / (np.mean(periods) + 1e-6)

    rms = librosa.feature.rms(y=audio, hop_length=HOP_LEN, frame_length=WIN_LEN)[0]
    voiced_idx = np.where(voiced_flag)[0]
    if len(voiced_idx) < 3:
        return jitter, 0.0
    rms_voiced = rms[voiced_idx]
    rms_diffs = np.abs(np.diff(rms_voiced))
    shimmer = np.mean(rms_diffs) / (np.mean(rms_voiced) + 1e-6)
    return jitter, shimmer

def extract_temporal_features(audio, sr=SR):
    """
    Извлекает временные признаки аудиозаписи.

    Аргументы:
        audio (np.ndarray): Аудиосигнал.
        sr (int): Частота дискретизации.

    Возвращает:
        np.ndarray: Массив признаков размера (число фреймов, 43).
    """
    mfcc_feats = extract_mfcc_with_deltas(audio, sr)
    f0_feats = extract_f0(audio, sr)
    rms_feats = extract_rms(audio, sr)
    hnr_feats = extract_hnr(audio, sr)
    zcr_feats = extract_zcr(audio, sr)
    temporal = np.concatenate([mfcc_feats, f0_feats, rms_feats, hnr_feats, zcr_feats], axis=1)
    return temporal  # (101, 43)

def extract_all_features_for_file(file_path):
    """
    Полное извлечение признаков для одного WAV-файла.

    Аргументы:
        file_path (str): Путь к WAV-файлу.

    Возвращает:
        tuple: (temporal_features (np.ndarray), jitter (float), shimmer (float))
    """
    audio, _ = librosa.load(file_path, sr=SR)
    target_len = SR
    if len(audio) > target_len:
        start = (len(audio) - target_len) // 2
        audio = audio[start:start + target_len]
    elif len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)), mode='constant')

    temporal = extract_temporal_features(audio, SR)
    jitter, shimmer = extract_jitter_shimmer(audio, SR)
    return temporal, jitter, shimmer


def process_all_in_one_dataset():
    """
    Обрабатывает все WAV-файлы из указанной директории и сохраняет итоговый датасет.
    Каждый элемент содержит признаки, джиттер, шиммер и метку класса (0, 1 или 2).

    Аргументы:
        base_path (str): Директория с входными WAV-файлами.
        output_dir (str): Директория для сохранения итогового датасета.

    Возвращает:
        int: Общее количество обработанных файлов.
    """
    
    base_path = "D:/Аудио выборка/Аудиосегменты/Предобработанные_данные"
    output_dir = "."
    os.makedirs(output_dir, exist_ok=True)

    # Маппинг суффиксов на цифровые метки класса
    suffixes_to_class_id = {
        "_0": 0,
        "_1": 1,
        "_2": 2,
    }

    files = list(Path(base_path).glob("*.wav"))
    print(f"Найдено файлов: {len(files)}")

    temporals = []
    jitters = []
    shimmers = []
    class_labels = []
    file_names = []

    for file_path in tqdm(files, desc="Обработка файлов"):
        file_name = file_path.stem
        class_id = None
        for suff, cid in suffixes_to_class_id.items():
            if file_name.endswith(suff):
                class_id = cid
                break
        if class_id is None:
            print(f"Пропущен файл без подходящего суффикса: {file_path.name}")
            continue
        try:
            temporal, jitter, shimmer = extract_all_features_for_file(str(file_path))
            temporals.append(temporal)
            jitters.append(jitter)
            shimmers.append(shimmer)
            class_labels.append(class_id)
            file_names.append(file_path.name)
        except Exception as e:
            print(f"Ошибка: {file_path.name} - {e}")

    total = len(class_labels)

    output_file = os.path.join(output_dir, "fatigue_features.npz")
    np.savez(
        output_file,
        temporals=np.array(temporals),
        jitters=np.array(jitters),
        shimmers=np.array(shimmers),
        labels=np.array(class_labels),
        filenames=np.array(file_names)
    )
    print(f"Датасет сохранён по пути: {output_file}")
    print(f"Всего обработано файлов: {total}")

    unique_classes, class_counts = np.unique(class_labels, return_counts=True)
    print("Количество примеров в каждом классе:")
    for cls, count in zip(unique_classes, class_counts):
        print(f"Класс {cls}: {count}")
    return total


process_all_in_one_dataset()

Найдено файлов: 3403


Обработка файлов: 100%|██████████| 3403/3403 [14:44<00:00,  3.85it/s]


Датасет сохранён по пути: .\fatigue_features.npz
Всего обработано файлов: 3403
Количество примеров в каждом классе:
Класс 0: 1546
Класс 1: 1247
Класс 2: 610


3403